# BTCUSDT 원본 ZIP 품질 검사

2025-01 ~ 2026-08, UTC 1분 봉. 원본 파일은 수정하지 않습니다. 코드 개선 작업의 재현용 노트북입니다.

OHLCV는 실제 호가·체결·펀딩 내역을 대체하지 않습니다. 원본 출처의 진위는 별도로 확인하지 않았습니다.

In [1]:
from pathlib import Path
import json, sys
ROOT = Path("/Users/yumin/Documents/trading")
sys.path.insert(0, str(ROOT))
manifest = json.loads((ROOT / "data/historical/btcusdt_2025_2026_verified/manifest.json").read_text())
summary = {key: manifest[key] for key in ["status", "file_count", "row_count", "unique_minutes", "start", "end_exclusive", "errors", "resampled_rows"]}
print(json.dumps(summary, indent=2))

{
  "status": "PASS",
  "file_count": 20,
  "row_count": 875520,
  "unique_minutes": 875520,
  "start": "2025-01-01T00:00:00+00:00",
  "end_exclusive": "2026-09-01T00:00:00+00:00",
  "errors": {},
  "resampled_rows": {
    "5m": 175104,
    "15m": 58368,
    "1h": 14592,
    "4h": 3648
  }
}


## 원본에서 검사 재실행

아래 셀은 같은 검사 함수를 원본 ZIP에 적용합니다. 재생성 데이터는 임시 폴더에 저장한 뒤 정리됩니다. 검사 구현은 `scripts/prepare_local_kline_archives.py`에 있습니다.

In [2]:
from tempfile import TemporaryDirectory
from scripts.prepare_local_kline_archives import prepare_archives
archives = [Path(item["path"]) for item in manifest["files"]]
with TemporaryDirectory(prefix="btc-quality-recheck-") as temporary:
    recheck = prepare_archives(archives, Path(temporary) / "dataset")
assert recheck["status"] == "PASS"
assert recheck["row_count"] == manifest["row_count"]
assert [item["sha256"] for item in recheck["files"]] == [item["sha256"] for item in manifest["files"]]
assert recheck["artifacts"] == manifest["artifacts"]
print("Source hashes, quality checks, and replay artifacts match.")

Source hashes, quality checks, and replay artifacts match.
